<a href="https://colab.research.google.com/github/viniciusarnom/estudo_dirigido/blob/main/Random%20Forest%20Classifica%C3%A7%C3%A3o%20de%20Label%20-%201000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Carregar os quatro arquivos CSV de features
print("Carregando arquivos...")
df_bes = pd.read_pickle("concatenated_features_bes_day1.bin").head(1000)
df_browning = pd.read_pickle("concatenated_features_browning_day1.bin").head(1000)
df_honors = pd.read_pickle("concatenated_features_honors_day1.bin").head(1000)
df_meb = pd.read_pickle("concatenated_features_meb_day1.bin").head(1000)

# 2. Atribuir uma Label única para cada localização/roteador
# Isso garante que o modelo saiba diferenciar as 4 origens
df_bes['Label'] = 0
df_browning['Label'] = 1
df_honors['Label'] = 2
df_meb['Label'] = 3

# 3. Concatenar todos os DataFrames em um único conjunto de dados
print("Concatenando os dados...")
df_final = pd.concat([df_bes, df_browning, df_honors, df_meb], ignore_index=True)

# 4. Preparar as Features (X) e o Alvo/Target (y)
X = df_final[['I', 'Q', 'Magnitude', 'Spectrum']]
y = df_final['Label']

# 5. Dividir os dados em Treino (70%) e Teste (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 6. Criar e treinar o modelo Random Forest
print("Treinando o modelo Random Forest Multiclasse (Isso pode levar alguns segundos)...")
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

# 7. Fazer as previsões usando os dados de teste invisíveis ao modelo
print("Calculando previsões e métricas...")
y_pred = rf_clf.predict(X_test)

# 8. Calcular as métricas
# Usamos average='weighted' para calcular a média ponderada entre as 4 classes
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# 9. Exibir os resultados globais
print("\n--- Resultados Globais da Classificação (4 Classes) ---")
print(f"Acurácia (Accuracy): {acc:.4f}")
print(f"Precisão (Precision): {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"F1-Score:            {f1:.4f}")

# 10. Exibir o relatório detalhado para ver o desempenho em CADA roteador separadamente
print("\n--- Relatório Detalhado por Roteador ---")
nomes_classes = ['bes (0)', 'browning (1)', 'honors (2)', 'meb (3)']
print(classification_report(y_test, y_pred, target_names=nomes_classes, zero_division=0))

Carregando arquivos...
Concatenando os dados...
Treinando o modelo Random Forest Multiclasse (Isso pode levar alguns segundos)...
Calculando previsões e métricas...

--- Resultados Globais da Classificação (4 Classes) ---
Acurácia (Accuracy): 0.4875
Precisão (Precision): 0.4814
Recall:              0.4875
F1-Score:            0.4797

--- Relatório Detalhado por Roteador ---
              precision    recall  f1-score   support

     bes (0)       0.36      0.34      0.35       315
browning (1)       0.72      0.65      0.68       307
  honors (2)       0.33      0.27      0.30       294
     meb (3)       0.52      0.71      0.60       284

    accuracy                           0.49      1200
   macro avg       0.48      0.49      0.48      1200
weighted avg       0.48      0.49      0.48      1200



In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier # Importação da Rede Neural
from sklearn.preprocessing import StandardScaler # Essencial para Redes Neurais
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Carregar os arquivos (Mantido conforme original)
print("Carregando arquivos...")
df_bes = pd.read_pickle("concatenated_features_bes_day1.bin").head(1000)
df_browning = pd.read_pickle("concatenated_features_browning_day1.bin").head(1000)
df_honors = pd.read_pickle("concatenated_features_honors_day1.bin").head(1000)
df_meb = pd.read_pickle("concatenated_features_meb_day1.bin").head(1000)

# 2. Atribuir Labels
df_bes['Label'] = 0
df_browning['Label'] = 1
df_honors['Label'] = 2
df_meb['Label'] = 3

# 3. Concatenar
print("Concatenando os dados...")
df_final = pd.concat([df_bes, df_browning, df_honors, df_meb], ignore_index=True)

# 4. Preparar Features (X) e Target (y)
X = df_final[['I', 'Q', 'Magnitude', 'Spectrum']]
y = df_final['Label']

# 5. Dividir os dados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- NOVIDADE: Normalização ---
# Redes neurais funcionam muito melhor quando os dados estão na mesma escala (média 0, desvio 1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. Criar e treinar o modelo MLP (Rede Neural)
print("Treinando a Rede Neural MLP (Isso pode levar alguns segundos)...")
# Configuração: 2 camadas escondidas de 100 e 50 neurônios, respectivamente.
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    activation='relu',
    solver='adam',
    random_state=42
)
mlp_clf.fit(X_train, y_train)

# 7. Fazer as previsões
print("Calculando previsões e métricas...")
y_pred = mlp_clf.predict(X_test)

# 8. Calcular as métricas (Mantido igual)
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# 9. Exibir os resultados globais
print("\n--- Resultados Globais da Classificação (4 Classes) ---")
print(f"Acurácia (Accuracy): {acc:.4f}")
print(f"Precisão (Precision): {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"F1-Score:            {f1:.4f}")

print("\n--- Relatório Detalhado por Roteador ---")
nomes_classes = ['bes (0)', 'browning (1)', 'honors (2)', 'meb (3)']
print(classification_report(y_test, y_pred, target_names=nomes_classes, zero_division=0))

Carregando arquivos...
Concatenando os dados...
Treinando a Rede Neural MLP (Isso pode levar alguns segundos)...
Calculando previsões e métricas...

--- Resultados Globais da Classificação (4 Classes) ---
Acurácia (Accuracy): 0.5275
Precisão (Precision): 0.5323
Recall:              0.5275
F1-Score:            0.5187

--- Relatório Detalhado por Roteador ---
              precision    recall  f1-score   support

     bes (0)       0.40      0.35      0.37       315
browning (1)       0.80      0.65      0.72       307
  honors (2)       0.41      0.32      0.36       294
     meb (3)       0.52      0.80      0.63       284

    accuracy                           0.53      1200
   macro avg       0.53      0.53      0.52      1200
weighted avg       0.53      0.53      0.52      1200

